# Exercise 9.1 — Variable importance, and a column of pure noise (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-01-importance.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Day 5 ended with a random forest that predicts diabetes better than a single
tree, but cannot be read the way a tree or a logistic regression can. The first
question everyone asks of such a model is: **which variables does it use?**
This exercise answers it in two different ways, which do not agree, and then
tests both with a predictor we know to be useless.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

------------------------------------------------------------------------

### Part 1 — The data and the forest

The 750 people in
[`data/diabetes_clean.csv`](data/diabetes_clean.csv), with every fifth person held back as
the test set. This time we keep five predictors: number of pregnancies,
glucose, BMI (`mass`), the diabetes pedigree function and age. Skin thickness
(`triceps`), insulin and blood pressure are left out, because in this file a
zero in those columns means "not measured", and too many of them are zero to
simply drop the rows.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

train["diabetes"].value_counts()

600 training rows, 220 with diabetes; 150 test rows, 53 with diabetes.

Fit a random forest of 500 trees on the training rows, exactly as on Day 5.
scikit-learn keeps a record of how much each variable helped while it grows
the trees; Part 2 reads it. A forest is random, so fix `random_state`.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_prob = rf.predict_proba(test[predictors])[:, 1]    # probability of diabetes

auc_before = roc_auc_score(test["diabetes"], test_prob)
round(auc_before, 3)

The test AUC is 0.824.

The two languages grow different random trees, so their forests, and every
number on this page that comes from them, differ slightly. The R tab gets an
AUC of 0.826.

> **Learning points**
>
> - We explain one model all day: a random forest of 500 trees on five predictors, fitted on the 600 training rows.
> - A forest is random: fix the seed, and expect a different forest (and slightly different numbers) in the other language.

------------------------------------------------------------------------

### Part 2 — Impurity-based importance

Every split in every tree makes the two groups it creates purer than the group
it started from: the Gini impurity goes down. **Impurity-based importance**
(also called Gini importance or *mean decrease in impurity*) is worked out like
this:

1. for each split in each tree, take the reduction in Gini impurity;
2. weight it by the number of training rows that reach that split;
3. add these up for each variable, over all the splits that used it;
4. average over the trees.

A variable that the trees use often, near the top, to make large clean splits
gets a high score. The forest computed all of this while it was being fitted,
so it costs nothing to ask for.

scikit-learn stores it in `feature_importances_`, one number per predictor in
the order of the columns. They are already scaled to add up to 1, so each one
is that variable's share.

> **Worked answer**

In [ ]:
impurity_importance = pd.Series(rf.feature_importances_, index=predictors)

impurity_importance = impurity_importance.sort_values(ascending=False)
print(impurity_importance.round(3))
print("sum:", impurity_importance.sum().round(3))

> R: glucose 0.358 of the total, then BMI 0.197, age 0.175, pedigree 0.170 and
> pregnancies 0.100. Python: glucose 0.311, then BMI 0.212, pedigree 0.195, age
> 0.179 and pregnancies 0.103; the shares add up to 1. Both forests put glucose first and
> pregnancies last; the three in between are close together and their order
> differs between the two forests.

> **Stuck?**
>
> Type `rf.` and look through the attributes that end in an underscore: they are
> the ones set by `.fit()`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
impurity_importance.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Mean decrease in impurity (share of the total)")
ax.set_title("Random forest: impurity-based importance")
fig.tight_layout()
plt.show()

**Question!** The impurity importance is computed from the splits the trees made
on the **training** rows. What could that mean for a variable the forest
happened to split on often, even though it does not really help to predict
diabetes in new people?

Keep your answer: Part 4 tests it.

> **Learning points**
>
> - Impurity-based importance adds up how much each variable reduced the Gini impurity over all the splits in the forest.
> - It is free, because the forest computes it while fitting, but it describes how the model used the training rows, not how well a variable predicts in new people.
> - Different packages scale it differently: ranger gives a raw total, scikit-learn shares that add up to 1.

------------------------------------------------------------------------

### Part 3 — Permutation importance

**Permutation importance** asks a different question: *if this variable carried
no information, how much worse would the model be on new people?* To find out,
take the test set, shuffle one column so that its values land on the wrong
patients, and score the model again. Nothing is refitted: the model stays the
same, only its input has been damaged. The importance is the drop in AUC.

Do it once by hand for glucose:

In [ ]:
rng = np.random.default_rng(1)

test_shuffled = test.copy()
test_shuffled["glucose"] = rng.permutation(test_shuffled["glucose"].values)   # glucose now belongs to someone else

auc_after = roc_auc_score(test_shuffled["diabetes"],
                          rf.predict_proba(test_shuffled[predictors])[:, 1])

print("before:", round(auc_before, 3))
print("after: ", round(auc_after, 3))
print("drop:  ", round(auc_before - auc_after, 3))

With glucose shuffled the AUC falls from 0.824 to 0.608, a drop of
0.216. **Your turn:** do the same for BMI (`mass`).

> **Worked answer**

In [ ]:
rng = np.random.default_rng(1)

test_shuffled = test.copy()
test_shuffled["mass"] = rng.permutation(test_shuffled["mass"].values)

auc_after = roc_auc_score(test_shuffled["diabetes"],
                          rf.predict_proba(test_shuffled[predictors])[:, 1])

print("drop:", round(auc_before - auc_after, 3))

> R: the AUC drops by 0.040. Python: by 0.043. Much less than for glucose. Run the
> cell again without `set.seed()` (or with a different seed) and the drop changes: one shuffle is
> one random draw.

Because one shuffle is one random draw, the proper version repeats it, 20 times
for each variable, and reports the average drop. scikit-learn's
`permutation_importance()` does exactly what you just did, for every predictor
in turn, with `scoring="roc_auc"`.

In [ ]:
perm = permutation_importance(rf, test[predictors], test["diabetes"],
                              scoring="roc_auc", n_repeats=20, random_state=1)

perm_importance = pd.DataFrame({"variable": predictors,
                                "importance": perm.importances_mean,
                                "sd": perm.importances_std})
perm_importance = perm_importance.sort_values("importance", ascending=False)
perm_importance.round(3)

In [ ]:
ordered = perm_importance.sort_values("importance")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(ordered["variable"], ordered["importance"], xerr=ordered["sd"],
        color="coral", capsize=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean drop in test AUC when the variable is shuffled (± 1 SD)")
ax.set_title("Random forest: permutation importance")
fig.tight_layout()
plt.show()

Glucose dominates: shuffling it costs the model 0.159 of AUC on average.
Everything else is small, and the error bars (one standard deviation over the
20 shuffles) are about as large as the differences between the variables.
The pedigree function (0.008) is within one standard deviation of zero, and
pregnancies (0.018) only just above it.

**Question!** Impurity importance gave BMI, age and the pedigree function
between 18% and 21% of the total each, 60% to 70% of glucose. Permutation
importance gives them a small fraction of glucose. Why can the two disagree so
much?

They measure different things. Impurity importance counts how much the trees
*used* a variable on the training rows. Permutation importance measures how
much the *predictions on new people* depend on it. A forest of deep trees uses
every variable a great deal to separate the training rows, including in ways
that do not carry over to new people. A permutation importance near zero, or
below it, means the model would do as well without the information in that
column.

> **Learning points**
>
> - Permutation importance is the drop in a test-set score (here the AUC) when one column is shuffled; the model is not refitted.
> - It must be repeated, because each shuffle is random; report the mean and the spread.
> - Impurity importance and permutation importance answer different questions, so their rankings can differ.

------------------------------------------------------------------------

### Part 4 — A control: a column of pure noise

How do we know which measure to believe? The way you would check any
measurement: measure something whose true value you already know. Add a
sixth predictor, `noise`, that is nothing but random numbers, drawn
independently of everything else. Whatever an importance measure says about
`noise`, the right answer is **zero**.

> **Worked answer**

In [ ]:
rng = np.random.default_rng(1)
train_noise = train.copy()
test_noise  = test.copy()
train_noise["noise"] = rng.normal(size=len(train_noise))
test_noise["noise"]  = rng.normal(size=len(test_noise))

predictors_noise = predictors + ["noise"]

rf_noise = RandomForestClassifier(n_estimators=500, random_state=1)
rf_noise = rf_noise.fit(train_noise[predictors_noise], train_noise["diabetes"])

pd.Series(rf_noise.feature_importances_, index=predictors_noise).round(3)

> R: `noise` gets 0.130 of the total, fifth of six: ahead of pregnancies
> (0.096) and close to the pedigree function (0.139). Python: `noise` gets
> 0.152, level with the pedigree function (0.152) and age (0.154), and ahead of
> pregnancies (0.097).

> **Stuck?**
>
> `size=` is how many numbers to draw: one per row, `len(train_noise)` for the
> training set and `len(test_noise)` for the test set.

The column of random numbers gets 15% of the total impurity importance: more
than the number of pregnancies, and as much as the pedigree function and age.
By this measure, `noise` is a respectable predictor.

Now the permutation importance of the same forest, on the test rows with their
own noise column:

In [ ]:
perm_noise = permutation_importance(rf_noise, test_noise[predictors_noise],
                                    test_noise["diabetes"], scoring="roc_auc",
                                    n_repeats=20, random_state=1)

perm_importance_noise = pd.DataFrame({"variable": predictors_noise,
                                      "importance": perm_noise.importances_mean,
                                      "sd": perm_noise.importances_std})
perm_importance_noise.sort_values("importance", ascending=False).round(3)

Permutation importance puts `noise` at −0.013, with a standard deviation of
0.007: slightly below zero, meaning that shuffling it made the predictions,
if anything, a little better. Glucose (0.152), BMI (0.031) and age (0.021)
stay clearly above it; the pedigree function (−0.004) does no better than
the noise.

**Question!** Go back to your answer to the question in Part 2. A variable can
get a sizeable impurity importance simply because it has many different values
to split on, and deep trees will find *some* split on it that tidies up a few
training rows. Random numbers have 600 different values. What does this tell
you about reading a ranking of impurity importances on its own?

A ranking always has a first and a last place, even when most of the variables
do nothing. A known-useless column gives you a floor: any variable that does
not clearly beat `noise` should not be described as important.

> **Learning points**
>
> - A column of pure noise is a control: its true importance is zero, so it shows what each measure reports for nothing.
> - Impurity importance gave the noise column a share comparable to real predictors; permutation importance on the test set put it at about zero.
> - Report permutation importance, with its spread, when the question is which variables matter for prediction.

## Stretch

> **Stretch — the forest's own permutation importance**
>
> ranger can compute a permutation importance itself while it fits, using the
> *out-of-bag* rows (the training rows each tree did not see). Refit the R forest
> with `set_engine("ranger", importance = "permutation")` and look at `vi()`. In
> Python there is no built-in equivalent, but you can run
> `permutation_importance()` on the training rows instead of the test rows. Does
> `noise` still come out at about zero? Why might permutation on the training
> rows be too generous to it?

> **Stretch — a noise column with only two values**
>
> Replace the normal noise with a random 0/1 column (R: `sample(0:1, n(), replace = TRUE)`;
> Python: `rng.integers(0, 2, size=...)`). What happens to its impurity
> importance, and why?